In [5]:
from PINN_XWave_DDP import *

In [6]:
#Global variables physics constants
vth = .1
omega_ce = 2.
omega = 2.5
omega_pe = 1.
phi_amp = 1.
Ay_amp = 1.
B_0_amp = 1.

k, k_lambda_D = dispersion(omega, omega_ce, omega_pe, vth)
print(f"k = {k:.3f}, k*lambda_D = {k_lambda_D:.3f}")

omega_list = [omega]
phi_amp_list = [phi_amp]  # Amplitude of the electric field
Ay_amp_list = [Ay_amp]
B0_list = [B_0_amp]
delta_list = [0.0]  # Phase factor for the wave solution

lamda = 2 * np.pi / k  # Wavelength
T = (2 * np.pi) / omega  # Period of the wave
xmin, xmax = 0, 3*lamda
tmin, tmax = 0, 3*T

L = xmax - xmin  # Length of the domain
tau = tmax - tmin  # Time duration of the wave
Nt = 2000 # Number of time points
Nx = 2000 # Number of spatial points
Nt_coll = 200 
Nx_coll = 200
dt = T/float(Nt)
dx = L/float(Nx)

t_arr = np.linspace(tmin, tmax, Nt)
x_arr = np.linspace(xmin, xmax, Nx)

print(f"k = {k:.4e}")
print(f"k lambda_debye = {k_lambda_D:.4e}")
print(f"L = {L:.3e} [c / omega_pe]")
print(f"T = {T:.3e} [1 / omega_pe]")
print(f"dx = {dx:.3e} [c / omega_pe]")
print(f"dt = {dt:.3e} [1 / omega_pe]")

X_arr, T_arr, phi_flat, Bdot_flat, Ax_flat, Ay_flat, Vx_flat, Vy_flat, N_flat, = generate_data(
xmin = xmin, xmax = xmax, tmin = tmin, tmax = tmax, nx = Nx, nt = Nt, omega_ce = omega_ce, omega_p = omega_pe, omega_list = omega_list, phi_amp_list=phi_amp_list, Ay_amp_list=Ay_amp_list, B0_list=B0_list, delta_list=delta_list)

x_sparse, t_sparse, phi_sparse, Bdot_sparse = sparse_measurements(X_arr, T_arr, phi_flat, Bdot_flat, num_samples=500)

x_coll, t_coll, dx, dt = collocation_points(xmin, xmax, tmin, tmax, Nx_coll, Nt_coll, L, tau)

k = 1.432, k*lambda_D = 0.143
k = 1.4318e+00
k lambda_debye = 1.4318e-01
L = 1.317e+01 [c / omega_pe]
T = 2.513e+00 [1 / omega_pe]
dx = 6.583e-03 [c / omega_pe]
dt = 1.257e-03 [1 / omega_pe]


NameError: name 'omega_pe' is not defined

In [ ]:
quantities_flat = [phi_flat, Bdot_flat, Ax_flat, Ay_flat, Vx_flat, Vy_flat, N_flat]
extent_GT = [xmin, xmax, tmin, tmax]
quantities_GT = [x.reshape(Nt, Nx) for x in quantities_flat]
titles_GT = [f'$\phi [\\frac{{m_{{e}}c^2}}{{e}}]$', 
            f'$\dot{{B}} [\\frac{{e}}{{m_{{e}}c}}]$',
            f'$A_x [\\frac{{m_{{e}}c^2}}{{e}}]$',
            f'$A_y [\\frac{{m_{{e}}c^2}}{{e}}]$',
            f'$E_x [\\frac{{\\omega_{{pe}}m_{{e}}c}}{{e}}]$ (longitudinal)', 
            f'$E_y [\\frac{{\\omega_{{pe}}m_{{e}}c}}{{e}}]$ (transverse)', 
            f'$B_z [\\frac{{e}}{{m_{{e}}c\\omega_{{pe}}}}]$',
            f'$v_x [c]$', 
            f'$v_y [c]$', 
            f'$N_e [n_0]$']

#Plot Ground Truth Values
print("-------Plotting Ground Truth Values-------------")
plot_analytical(quantities=quantities_GT, titles = titles_GT, extent = extent_GT)

<>:4: SyntaxWarning: invalid escape sequence '\p'
<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\p'
<>:5: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_1721686/1754455370.py:4: SyntaxWarning: invalid escape sequence '\p'
  titles_GT = [f'$\phi [\\frac{{m_{{e}}c^2}}{{e}}]$',
/tmp/ipykernel_1721686/1754455370.py:5: SyntaxWarning: invalid escape sequence '\d'
  f'$\dot{{B}} [\\frac{{e}}{{m_{{e}}c}}]$',
/tmp/ipykernel_1721686/1754455370.py:4: SyntaxWarning: invalid escape sequence '\p'
  titles_GT = [f'$\phi [\\frac{{m_{{e}}c^2}}{{e}}]$',
/tmp/ipykernel_1721686/1754455370.py:5: SyntaxWarning: invalid escape sequence '\d'
  f'$\dot{{B}} [\\frac{{e}}{{m_{{e}}c}}]$',


NameError: name 'phi_flat' is not defined

In [ ]:
#Plot constraints
phi = quantities_GT[0]
Bdot= quantities_GT[1]
Ax = quantities_GT[2]
Ay = quantities_GT[3]
Ex = quantities_GT[4]
Ey = quantities_GT[5]
Bz = quantities_GT[6]
Vx = quantities_GT[7]
Vy = quantities_GT[8]
N = quantities_GT[9]


phi_x = derivative_x(phi, dx)
phi_xx = derivative_x(phi_x, dx)
phi_t = derivative_t(phi, dt)
phi_tt = derivative_t(phi_t, dt)

Ax_x = derivative_x(Ax, dx)
Ax_xx = derivative_x(Ax_x, dx)
Ax_t = derivative_t(Ax, dt)
Ax_tt = derivative_t(Ax_t, dt)
Ay_x = derivative_x(Ay, dx)
Ay_xx = derivative_x(Ay_x, dx)
Ay_t = derivative_t(Ay, dt)
Ay_tt = derivative_t(Ay_t, dt)

Ex_x = derivative_x(Ex, dx)
Ex_t = derivative_t(Ex, dt)
Ey_x = derivative_x(Ey, dx)
Ey_t = derivative_t(Ey, dt)

Bz_x = derivative_x(Bz, dx)
Bz_t = derivative_t(Bz, dt)

gauss_x = Ey_x - N
electric_field_x = Ex + phi_x + Ax_t
electric_field_y = Ey + Ay_t
Bfield_curl_A = Bz - Ay_x
faraday_y = -Bz_x + Vy + Ey_t
faraday_x = Vx + Ex_t
ampere_z = Ey_x + Bdot
coul_gauge_x = Ax_x + phi_t
wave_A_x = Ax_tt - Ax_xx + Vx
wave_A_y = Ay_tt + Vy
wave_phi = phi_tt - phi_xx + N
Bdot_def = Bz_t - Bdot

constraints_ext = [x_arr[1], x_arr[-2], t_arr[1], t_arr[-2]]
constraints = [gauss_x, electric_field_x, electric_field_y, Bfield_curl_A,
               faraday_y, faraday_x, ampere_z, coul_gauge_x,
               wave_A_x, wave_A_y, wave_phi, Bdot_def]
constraint_titles = [f'$|\\partial_x E_y - N_e|$', 
          f'$|E_x + \\phi_x + \\partial_t A_x|$',
          f'$|E_y + \\partial_t A_y|$',
          f'$|B_z - \\partial_x A_y|$',
          f'$|-\\partial_x B_z + V_y + \\partial_t E_y|$',
          f'$|V_x + \\partial_t E_x|$',
          f"$|\\partial_x E_y + \\dot{{B}}|$",
          f'$|\\partial_x A_x + \\partial_t \\phi|$',
          f'$|\\partial_{{t}}^{{2}} A_x - \\partial_{{x}}^{{2}} A_x + V_x|$',
          f'$|\\partial_{{t}}^{{2}} A_y + V_y|$',
          f'$|\\partial_{{t}}^{{2}} \\phi - \\partial_{{x}}^{{x}} \\phi + N_e|$',
          f'$|\\partial_t B - \\dot{{B}}|']